# QPKI-SmartHome — Demo Notebook

**Paper:** QPKI-SmartHome: Post-Quantum Public Key Infrastructure with Lightweight Lattice-Based Authentication for Cloud-Tethered IoT Smart Home Devices  
**Journal:** MDPI Sensors, 2026  

This notebook walks through:
1. Loading and preprocessing the CIC-IoT-2023 dataset
2. Training the ThreatClassifier (DNN)
3. Evaluating classification metrics
4. Running the energy-aware scheduler simulation
5. Visualizing all paper results

In [ ]:
import sys, os
sys.path.insert(0, os.path.join(os.getcwd(), '..'))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch

from config import ML_KEM, ML_DSA, BASELINES, PERFORMANCE_RESULTS, CERTIFICATE_COMPRESSION
from model import ThreatClassifier, EnergyAwareScheduler, QPKIHandshakeSimulator, DeltaEncodingCompressor
from utils import (
    plot_scheduler_heatmap, plot_cert_compression,
    plot_latency_breakdown, plot_pki_hierarchy,
    build_certificate_chain_info
)

print('Setup complete. PyTorch', torch.__version__)

## 1. System Overview (Section 3)

In [ ]:
# Section 4.1 — ML-KEM parameter sets
print('ML-KEM Parameter Sets (NIST FIPS 203):')
print(f'{"Parameter Set":<16} {"Security":<10} {"PK (B)":<10} {"CT (B)":<10} {"Energy (mJ)":<12}')
print('-' * 60)
for name, p in ML_KEM.items():
    print(f'{name:<16} {p["security_level"]:<10} {p["pk_bytes"]:<10} {p["ct_bytes"]:<10} {p["energy_mJ"]:<12}')

print()
print('ML-DSA Parameter Sets (NIST FIPS 204):')
print(f'{"Parameter Set":<16} {"Security":<10} {"PK (B)":<10} {"Sig (B)":<10} {"Role"}')
print('-' * 60)
for name, p in ML_DSA.items():
    print(f'{name:<16} {p["security_level"]:<10} {p["pk_bytes"]:<10} {p["sig_bytes"]:<10} {p["role"]}')

## 2. Certificate Compression (Section 4.2)

In [ ]:
compressor = DeltaEncodingCompressor()
chain = build_certificate_chain_info()
result = compressor.compress_chain(chain)

print('Certificate Chain Compression:')
for k, v in result.items():
    print(f'  {k:<30s}: {v}')

# Reproduce Table 3
print()
print('Table 3 — ML-DSA-65 Intermediate CA Certificate:')
print(f'  Full DER:              {CERTIFICATE_COMPRESSION["full_der_bytes"]:>6} bytes')
print(f'  After delta strip:     {CERTIFICATE_COMPRESSION["after_delta_strip_bytes"]:>6} bytes')
print(f'  After Zstd coding:     {CERTIFICATE_COMPRESSION["after_zstd_bytes"]:>6} bytes')
print(f'  ECDSA-P256 baseline:   {CERTIFICATE_COMPRESSION["ecdsa_p256_cert_bytes"]:>6} bytes')
print(f'  Compression ratio:     {CERTIFICATE_COMPRESSION["compression_ratio"]*100:.0f}%')

In [ ]:
# Plot certificate compression pipeline
path = plot_cert_compression('../results/cert_compression.png')
from IPython.display import Image
Image(path)

## 3. Energy-Aware Scheduler (Section 4.3)

In [ ]:
sched = EnergyAwareScheduler()

print('Scheduler decision table:')
print(f'{"Battery":>10} {"Threat":>8} {"Selected KEM":<16} {"Energy (mJ)"}')
print('-' * 50)
for batt in [0.10, 0.35, 0.60, 0.80]:
    for threat in [0.20, 0.85]:
        kem = sched.select_kem(batt, threat)
        e   = sched.get_energy(kem)
        print(f'{batt:>10.0%} {threat:>8.0%} {kem:<16} {e:.2f}')

wavg = sched.weighted_avg_energy()
print(f'\nWeighted avg energy: {wavg:.3f} mJ  (paper reports 4.749 mJ)')

In [ ]:
# Scheduler heatmap
path = plot_scheduler_heatmap('../results/scheduler_heatmap.png')
Image(path)

## 4. Handshake Performance Simulation (Section 6.2)

In [ ]:
sim = QPKIHandshakeSimulator()

print('Handshake Simulation (Table 4):')
print(f'{"KEM":<14} {"Comp (ms)":>10} {"E2E (ms)":>10} {"Energy (mJ)":>12} {"RAM (KB)":>9}')
print('-' * 58)
for kem in ML_KEM:
    r = sim.simulate(kem, 'ML-DSA-44')
    print(f'{kem:<14} {r["computation_latency_ms"]:>10.1f} {r["end_to_end_latency_ms"]:>10.1f} '
          f'{r["energy_mJ"]:>12.3f} {r["peak_ram_bytes"]/1024:>9.1f}')

print()
print('Paper reported values (ML-KEM-768 + ML-DSA-44):')
for k, v in PERFORMANCE_RESULTS.items():
    print(f'  {k:<40s} {v}')

In [ ]:
# Latency breakdown
path = plot_latency_breakdown('../results/latency_breakdown.png')
Image(path)

## 5. PKI Hierarchy (Section 4.1)

In [ ]:
path = plot_pki_hierarchy('../results/pki_hierarchy.png')
Image(path)

## 6. Baseline Comparison (Section 6.3, Table 5)

In [ ]:
df_baselines = pd.DataFrame(BASELINES).T.reset_index()
df_baselines.columns = ['Method', 'Energy (mJ)', 'Latency (ms)', 'Cert Size (KB)']
print(df_baselines.to_string(index=False))

# Energy reduction vs PQ-TLS-IoT
pqtls_e = BASELINES['PQ-TLS-IoT']['energy_mJ']
our_e   = BASELINES['QPKI-SmartHome']['energy_mJ']
print(f'\nEnergy reduction vs PQ-TLS-IoT: {(1 - our_e/pqtls_e)*100:.1f}%  (paper: 74%)')

## 7. Load Data and Train (requires CIC-IoT-2023)

In [ ]:
# Uncomment to download and train:
# !python ../train.py --sample-frac 0.05 --epochs 5
# After training, evaluate:
# !python ../evaluate.py
print('To train: run  python train.py  from the repo root.')
print('To evaluate: run  python evaluate.py')

## 8. Model Architecture

In [ ]:
model = ThreatClassifier(input_dim=20, n_classes=2)
print(model)
print(f'\nTotal trainable parameters: {model.parameter_count():,}')

# Forward pass test
x = torch.randn(4, 20)
out = model(x)
print(f'Output shape: {out.shape}  (batch=4, n_classes=2)')
threat_scores = model.predict_threat_score(x)
print(f'Threat scores: {threat_scores.numpy()}')